In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from phd_project.config import config
from phd_project.scripts.case_study_design_scripts.design_site_mdof import (
    design_sites_parallel,
    default_n_workers,
)
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_structural_model,
    copy_analysis_config,
    copy_batch_ida_buildings,
    copy_file,
)
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_primary_modes_from_design_file,
    get_n_damping_modes_from_design_file,
)
from phd_project.scripts.loading_protocols import FEMA_461_loading_protocol

cfg = config.load_config()

In [2]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Destination root for the (large) analysis folders -- set this to your external
# drive. Folders are written as DEST_ROOT/site_{ii}/sdof_param/ and .../mdof/.
DEST_ROOT = cfg["analysis_data"]["reference_structures_dc2_scA"]

# --- structure / design (fixed; matches 01_design_case_study_structures.py) ---
SITE_CATEGORY = "A"        # soil class for BOTH the MDOF design and SDOF spectrum
N_STOREYS = 3
DUCTILITY_CLASS = 2
STOREY_HEIGHT = 3500       # mm
BAY_WIDTH = 7000           # mm
MAX_DESIGN_ITERS = 15

S_ALPHA_RPS = [0.133, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0, 1.2]

# --- parallelism ---
N_WORKERS = default_n_workers()    # max(cpu_count - 3, 1)

SETUP_ANALYSES = True

if SETUP_ANALYSES:
    # --- cyclic pushover (FEMA 461) ---
    CPO_U_MAX = 250          # mm, peak cyclic amplitude (+/-)
    CPO_N_STEPS = 12         # FEMA 461 amplitude steps (12 -> matches example sequence)
    CPO_DU = 0.2             # mm, base displacement ramp step (dU_max)
    CPO_DISPLACEMENTS = np.round(
        np.append(FEMA_461_loading_protocol(CPO_U_MAX, CPO_N_STEPS), 0), 3
    ).tolist()

    # --- analysis ---
    # NOTE: the MSA parameters (GM_SETS, MAX_N_RECORDS, STRIPE_ORDER_ASCENDING) live in
    # 050_setup_msa_runs_for_complete_sites, which owns the MSA wiring.
    DAMPING_RATIO = 0.05
    DISP_LIMIT = 700           # mm; SDOF collapse displacement limit
    MDOF_DRIFT_LIMIT = 0.2     # MDOF collapse drift limit

In [3]:
design_root = Path(cfg["models"]["casestudy_designs_dc2_scA_reference"])
design_root.mkdir(parents=True, exist_ok=True)
summary_path = design_root / "site_designs_summary.csv"

tags = [f"3s_dc2_scA_{int(salpha*100)}" for salpha in S_ALPHA_RPS]

jobs = [(salpha, tags[ii], design_root) for ii, salpha in enumerate(S_ALPHA_RPS)]

# only (re)design structures whose tag is not already in the summary csv
existing_df = pd.read_csv(summary_path, index_col="tag") if summary_path.exists() else None
done_tags = set(existing_df.index) if existing_df is not None else set()
pending_jobs = [j for j in jobs if j[1] not in done_tags]

if pending_jobs:
    print(f"designing {len(pending_jobs)} of {len(jobs)} structures "
          f"({len(jobs) - len(pending_jobs)} already in {summary_path.name})")
    design_results = design_sites_parallel(
        pending_jobs,
        n_workers=N_WORKERS,
        site_category=SITE_CATEGORY,
        ductility_class=DUCTILITY_CLASS,
        storey_height=STOREY_HEIGHT,
        n_storeys=N_STOREYS,
        max_iters=MAX_DESIGN_ITERS,
    )
    new_df = pd.DataFrame(design_results).set_index("tag")
    if existing_df is not None:
        existing_df = existing_df.drop(index=new_df.index, errors="ignore")
        designs_df = pd.concat([existing_df, new_df])
    else:
        designs_df = new_df
    designs_df = designs_df.sort_index()
    designs_df.to_csv(summary_path)
else:
    print(f"all {len(jobs)} structures already in {summary_path.name}; skipping design")
    designs_df = existing_df.sort_index()

designs = designs_df.to_dict("index")

failed = designs_df[~designs_df["success"].fillna(False)]
if len(failed):
    print(f"WARNING: {len(failed)} designs did not succeed:\n{failed.index.tolist()}")

designs_df[["S_alpha_RP", "Vb_coeff", "Wt", "T", "q_design", "success"]]

all 9 structures already in site_designs_summary.csv; skipping design


,S_alpha_RP,Vb_coeff,Wt,T,q_design,success
tag,,,,,,
3s_dc2_scA_100,1.000,0.416667,3061800.0,0.396737,2.040000,True
3s_dc2_scA_120,1.200,0.500000,3061800.0,0.369427,2.040000,True
3s_dc2_scA_13,0.133,0.026098,3061800.0,0.719978,1.415665,True
3s_dc2_scA_20,0.200,0.027234,3061800.0,0.719978,2.040000,True
3s_dc2_scA_30,0.300,0.043757,3061800.0,0.672157,2.040000,True
3s_dc2_scA_40,0.400,0.086012,3061800.0,0.581311,2.040000,True
3s_dc2_scA_50,0.500,0.114445,3061800.0,0.546113,2.040000,True
3s_dc2_scA_60,0.600,0.140632,3061800.0,0.533308,2.040000,True
3s_dc2_scA_70,0.700,0.257673,3061800.0,0.452771,2.040000,True


In [5]:
if SETUP_ANALYSES:
    def add_cyclic_pushover_files(folder: Path, ctrl_node: int) -> Path:
        """Add a FEMA 461 cyclic pushover run script + config to an analysis folder.
        The config imports structural_model/model_init from the folder itself."""
        copy_file(cfg["templates"]["run_cyclic_pushover"], folder / "run_cyclic_pushover.py")
        cfg_dst = folder / "config_cyclic_pushover.py"
        copy_analysis_config(
            cfg["templates"]["config_cyclic_pushover"],
            cfg_dst,
            results_folder_name="cyclic_pushover",
            update_config={
                "displacement_type": "displacement",
                "dU": CPO_DU,
                "ctrl_node": ctrl_node,
                "displacements": CPO_DISPLACEMENTS,
            },
        )
        return cfg_dst

    def build_mdof_folder(tag: int) -> Path:
        folder = DEST_ROOT / f"{tag}" / "mdof"
        folder.mkdir(parents=True, exist_ok=True)

        # design file (structural_model reads it from its own folder)
        design_out = Path(designs[tag]["design_out_json"])
        design_dst = folder / f"{tag}_designfile.json"
        copy_file(design_out, design_dst)

        # structural model
        n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
        copy_structural_model(
            cfg["templates"]["structural_model"],
            folder / "structural_model.py",
            design_json=design_dst.name,
            damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
            recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
        )

        # modal analysis (for participation factor)
        copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
        copy_analysis_config(
            cfg["templates"]["config_modal"],
            folder / "config_modal.py",
            results_folder_name="modal",
            update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
        )

        # cyclic pushover (FEMA 461)
        add_cyclic_pushover_files(folder, get_control_node_from_design_file(design_dst))
        
        # ida analysis
        # copy files that don't need changing:
        src_proc = cfg["templates"]["ida_process_recorder_roof_drift"]
        dst_proc = folder / "ida_process_recorders.py"
        copy_file(src_proc, dst_proc)

        src_im = cfg["templates"]["config_im_SA"]
        dst_im = folder / "config_im.py"
        copy_file(src_im, dst_im)

        src_inj = cfg["templates"]["nltha_injection_update_damping"]
        dst_inj = folder / "injection_functions.py"
        copy_file(src_inj, dst_inj)

        src = cfg["templates"]["run_batch_ida_per_record"]
        dst = folder / "run_batch_ida_per_record.py"
        copy_file(src, dst)

        src = cfg["templates"]["run_ida_htf_per_record"]
        dst = folder / "run_ida_htf_per_record.py"
        copy_file(src, dst)

        # now copy the config 
        src_config = cfg["templates"]["config_ida_htf"]
        dst_ida_config = folder / "config_ida_htf_femap695.py"

        changes = {
            "results_folder_name": "ida_femap695",
            "gm_json_src_str" : 'D:/gm_records_p695',
            "record_filenames": ['fema_p695_120111.json',
                                'fema_p695_120121.json',
                                'fema_p695_120411.json',
                                'fema_p695_120521.json',
                                'fema_p695_120611.json',
                                'fema_p695_120621.json',
                                'fema_p695_120711.json',
                                'fema_p695_120721.json',
                                'fema_p695_120811.json',
                                'fema_p695_120821.json',
                                'fema_p695_120911.json',
                                'fema_p695_120921.json',
                                'fema_p695_121011.json',
                                'fema_p695_121021.json',
                                'fema_p695_121111.json',
                                'fema_p695_121211.json',
                                'fema_p695_121221.json',
                                'fema_p695_121321.json',
                                'fema_p695_121411.json',
                                'fema_p695_121421.json',
                                'fema_p695_121511.json',
                                'fema_p695_121711.json',
                                ],
        }

        copy_analysis_config(src_config, dst_ida_config, **changes)

        return folder, dst_ida_config


    mdof_folders = {tag: build_mdof_folder(tag) for tag in tqdm(
        [tag for tag in tags if designs[tag]["success"]],
        desc="Building MDOF folders")}
    print(f"built {len(mdof_folders)} MDOF folders")

    # copy and configure a batch file
    folders_and_configs = [{"folder":v[0], "config":v[1]} for v in mdof_folders.values()]
    
    src = cfg["templates"]["run_batch_ida_buildings"]
    dst = DEST_ROOT / "run_batch_idas.py"

    copy_batch_ida_buildings(src, dst, folders_and_configs)
    

Building MDOF folders:   0%|          | 0/9 [00:00<?, ?it/s]

built 9 MDOF folders
